# 03 — House PDF quality & smoke test

## Objectif

Mesurer la qualité technique des PDF House téléchargés.

Produire :
- `house_pdf_text_quality.csv` ;
- `house_sample_extraction_smoke_test.csv`.

In [ ]:
from pathlib import Path
import sys

import pandas as pd
from tqdm.auto import tqdm

ROOT = Path.cwd().resolve()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))

from src.pdf_quality import (
    run_pdf_quality_audit, summarize_pdf_quality, choose_smoke_sample,
    run_smoke_test, append_quality_report,
)
from src.utils import AUDIT_DIR, require_file

print("Imports OK")

In [ ]:
MAX_FILES = None  # Mettre 50 pour un audit rapide.
manifest_path = AUDIT_DIR / "house_pdf_manifest.csv"
quality_path = AUDIT_DIR / "house_pdf_text_quality.csv"
smoke_path = AUDIT_DIR / "house_sample_extraction_smoke_test.csv"

In [ ]:
require_file(manifest_path, "Run 02_house_pdf_download_manifest.ipynb first.")
manifest = pd.read_csv(manifest_path, dtype={"doc_id": str})

print("manifest shape:", manifest.shape)
display(manifest["download_status"].value_counts(dropna=False))
display(manifest[["year", "n_pages"]].describe())

In [ ]:
quality = run_pdf_quality_audit(
    manifest,
    output_path=quality_path,
    max_files=MAX_FILES,
)

print("quality shape:", quality.shape)
display(quality.head())

In [ ]:
summary = summarize_pdf_quality(quality)
summary

In [ ]:
display(quality["quality_status"].value_counts(dropna=False))
problem_cols = ["year", "doc_id", "quality_status", "text_chars_first_pages", "error_message"]
display(quality[quality["quality_status"] != "ok_text"][problem_cols].head(20))

In [ ]:
sample = choose_smoke_sample(manifest, quality, random_state=42)
print("sample shape:", sample.shape)
display(sample[["year", "doc_id", "declarant_name_raw", "n_pages"]].head())

In [ ]:
smoke = run_smoke_test(sample, output_path=smoke_path)
print("smoke shape:", smoke.shape)
display(smoke.head())

In [ ]:
report_path = append_quality_report(summary)
print("Written:", quality_path.relative_to(ROOT))
print("Written:", smoke_path.relative_to(ROOT))
print("Updated:", report_path.relative_to(ROOT))

## Conclusion

**OK** si les PDF sont majoritairement `ok_text` et que les cas faibles sont listés.

**NEXT** — lancer le diagnostic Quiver avec `04_quiver_access_validation_2024.ipynb`.